In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
def process_list(row):
    if row is None:
        return []
    else:
        return [int(item) for item in row if isinstance(item, str) and item.isdigit()]

In [4]:
# Distribution Rules

with open(data_dir / "metadata.json") as f:
    metadata = json.load(f)

with open(data_dir / "distribution_rules.json") as f:
    distribution_rules = json.load(
        f, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj}
    )

rules = distribution_rules.get("rules", [])

for rule in rules:
    rrg = rule.get("rrg")
    if rrg is None:
        rule["rrg"] = {}

    for key in list(rule.keys()):
        if key in metadata:
            rule[metadata[key]] = rule.pop(key)

    level_mapping = metadata["level_mapping"]
    for key, value in rule.items():
        if isinstance(value, dict):
            for k in list(value.keys()):
                if k in level_mapping:
                    value[level_mapping[k]] = value.pop(k)

df_rules = pd.json_normalize(rules, sep="_")
df_rules["credential_list"] = df_rules["credential_list"].apply(process_list)
df_rules["hotel_list"] = df_rules["hotel_list"].apply(process_list)

In [5]:
# Credential Group

with open(data_dir / "credential_group.json") as f:
    credential_group = json.load(f)

df_credential_group = pd.DataFrame(credential_group.get("clientGroupList", []))
df_credential_group.rename(columns={"clientCodes": "credential_list"}, inplace=True)
df_credential_group["id"] = df_credential_group["id"].astype(int)
df_credential_group["credential_list"] = df_credential_group["credential_list"].apply(
    process_list
)

In [16]:
df_rules_credential = df_rules.explode("credential_list")[
    ["id", "credential_choices", "credential_list"]
]

In [30]:
with pd.option_context("display.max_columns", None):
    display(df_rules[df_rules["destination_choices"] != 0].head())

,refundable,rate,max_release,dynamic_commission,age,edit_state,obsolete,name,description,updated_by,updated_on,tag,id,range_choices,range_from,range_to,credential_choices,credential_list,ps_choices,ps_list,provider_choices,provider_list,hotel_choices,hotel_list,destination_choices,destination_list,market_choices,market_list,meal_choices,meal_list,check_in_choices,check_in_from,check_in_to,booking_date_choices,booking_date_from,booking_date_to,days_of_week_choices,days_of_week_list,room_choices,room_list,num_of_nights_choices,num_of_nights_list,hours_choices,hours_list
9,0,0,0,False,0,1,False,TRVM-Agoda US,15.03.23-close US- Agoda. Ref. Raquel /Gemma,raquel.yague@smyrooms.com,26-04-2023 19:57:32,1,394640,0.0,0.0,0.0,1,"[5226, 8550, 11945]",0,None,1,[TRVM],0,[],2,[US],0,None,0,None,0,None,None,0,None,None,0,None,0,None,0,None,0,None
102,0,0,0,False,0,1,False,Cierre Sembo SHS,High % of quote errors with supplier SHS + mar...,patricia.benitez@smyrooms.com,21-08-2024 15:18:23,0,611203,0.0,0.0,0.0,1,[11896],0,None,1,[SHS],0,[],2,[AE],1,[SE],0,None,0,None,None,0,None,None,0,None,0,None,0,None,0,None
130,0,2,0,False,0,1,False,B2B rates on UK market,Closing B2B rates on UK market to avoid violat...,daniel.diez@smyrooms.com,19-04-2023 13:04:54,0,394770,0.0,0.0,0.0,1,[4321],0,None,0,None,0,[],2,"[GB, IE]",0,None,0,None,0,None,None,0,None,None,0,None,0,None,0,None,0,None
146,0,0,0,False,0,1,True,TVLK - PK to ID,Juan 19/08/24 Close ID to PK fare due errors,juan.pinzon@smyrooms.com,19-08-2024 10:47:51,4,607485,0.0,0.0,0.0,1,[8321],0,None,1,[PKF],0,[],2,[ID],0,None,0,None,0,None,None,1,2024-08-19,2024-08-25,0,None,0,None,0,None,0,None
151,0,0,0,False,0,1,False,"HB Sensitive Close Expedia for destinations CN,TH",HB Sensitive Close Expedia for destinations CN...,AzureFunctions,10-09-2024 11:57:16,0,394044,0.0,0.0,0.0,1,[11656],0,None,1,"[EXPRTA, EXPR, EXPRH, EXPRDS, EXPRDSH]",0,[],2,"[CN, TH]",0,None,0,None,0,None,None,0,None,None,0,None,0,None,0,None,0,None


In [27]:
for x in df_rules["credential_list"]:
    print(x)
    break

[5226, 8550]
